In [1]:
year = 2007
month = 1

In [2]:
# Parameters
year = 2003
month = 2


In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap
import calendar
import datetime
from datetime import date

### URLs

In [4]:
# Ufiles = "https://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Ufiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Vfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridV"
Wfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridW"
Tfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridT"
Sfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridS"
# #mesh url
# Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
# Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [5]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [6]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [7]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [8]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

(2305, 3565, 1374, 1873)

In [9]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

2003-02-28


In [10]:
def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [11]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [12]:
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 2003-02-01 12:00:00
end_date 2003-02-02 12:00:00
start_date 2003-02-03 12:00:00
end_date 2003-02-04 12:00:00
start_date 2003-02-05 12:00:00
end_date 2003-02-06 12:00:00
start_date 2003-02-07 12:00:00
end_date 2003-02-08 12:00:00
start_date 2003-02-09 12:00:00
end_date 2003-02-10 12:00:00
start_date 2003-02-11 12:00:00
end_date 2003-02-12 12:00:00
start_date 2003-02-13 12:00:00
end_date 2003-02-14 12:00:00
start_date 2003-02-15 12:00:00
end_date 2003-02-16 12:00:00
start_date 2003-02-17 12:00:00
end_date 2003-02-18 12:00:00
start_date 2003-02-19 12:00:00
end_date 2003-02-20 12:00:00
start_date 2003-02-21 12:00:00
end_date 2003-02-22 12:00:00
start_date 2003-02-23 12:00:00
end_date 2003-02-24 12:00:00
start_date 2003-02-25 12:00:00
end_date 2003-02-26 12:00:00
start_date 2003-02-27 12:00:00
end_date 2003-02-28 12:00:00


### Data download

In [13]:
U_out = f'U_{start_date.strftime("%Y-%m")}.nc'
V_out = f'V_{start_date.strftime("%Y-%m")}.nc'
W_out = f'W_{start_date.strftime("%Y-%m")}.nc'
T_out = f'T_{start_date.strftime("%Y-%m")}.nc'
S_out = f'S_{start_date.strftime("%Y-%m")}.nc'

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables/'

In [14]:
#U 
download_MERCATOR(
    Ufiles, "vozocrtx", starts, ends, x0, x1, y0, y1,outpath+U_out
)

  0%|                                                          | 0/14 [00:00<?, ?it/s]

  7%|███▌                                             | 1/14 [01:51<24:10, 111.55s/it]

 14%|███████▏                                          | 2/14 [02:30<13:43, 68.59s/it]

 21%|██████████▋                                       | 3/14 [02:49<08:29, 46.35s/it]

 29%|██████████████▎                                   | 4/14 [03:09<05:58, 35.88s/it]

...

 86%|██████████████████████████████████████████       | 12/14 [06:12<00:45, 22.91s/it]

 93%|█████████████████████████████████████████████▌   | 13/14 [06:34<00:22, 22.63s/it]

100%|█████████████████████████████████████████████████| 14/14 [07:07<00:00, 25.80s/it]

100%|█████████████████████████████████████████████████| 14/14 [07:07<00:00, 30.52s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/U_2003-02.nc


In [15]:
download_MERCATOR(
    Vfiles, "vomecrty", starts, ends, x0, x1, y0, y1,outpath+V_out 
)

  0%|                                                          | 0/14 [00:00<?, ?it/s]

  7%|███▌                                              | 1/14 [01:11<15:35, 71.95s/it]

 14%|███████▏                                          | 2/14 [01:31<08:11, 40.99s/it]

 21%|██████████▋                                       | 3/14 [02:55<11:05, 60.54s/it]

 29%|██████████████▎                                   | 4/14 [03:21<07:51, 47.13s/it]

...

 86%|██████████████████████████████████████████       | 12/14 [06:45<00:51, 25.51s/it]

 93%|█████████████████████████████████████████████▌   | 13/14 [07:06<00:23, 24.00s/it]

100%|█████████████████████████████████████████████████| 14/14 [07:33<00:00, 24.91s/it]

100%|█████████████████████████████████████████████████| 14/14 [07:33<00:00, 32.37s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/V_2003-02.nc


In [16]:
download_MERCATOR(
    Wfiles, "vovecrtz", starts, ends, x0, x1, y0, y1,outpath+W_out 
)

  0%|                                                          | 0/14 [00:00<?, ?it/s]

  7%|███▌                                              | 1/14 [00:53<11:36, 53.59s/it]

 14%|███████▏                                          | 2/14 [01:14<06:52, 34.37s/it]

 21%|██████████▋                                       | 3/14 [01:34<05:07, 27.96s/it]

 29%|██████████████▎                                   | 4/14 [01:56<04:16, 25.61s/it]

...

 86%|██████████████████████████████████████████       | 12/14 [06:53<01:51, 55.67s/it]

 93%|█████████████████████████████████████████████▌   | 13/14 [09:05<01:18, 78.73s/it]

100%|█████████████████████████████████████████████████| 14/14 [09:31<00:00, 62.71s/it]

100%|█████████████████████████████████████████████████| 14/14 [09:31<00:00, 40.81s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/W_2003-02.nc


In [17]:
download_MERCATOR(
    Tfiles, "votemper", starts, ends, x0, x1, y0, y1,outpath+T_out
)

  0%|                                                          | 0/14 [00:00<?, ?it/s]

  7%|███▌                                             | 1/14 [02:45<35:56, 165.85s/it]

 14%|███████                                          | 2/14 [04:09<23:33, 117.77s/it]

 21%|██████████▋                                       | 3/14 [04:30<13:27, 73.42s/it]

 29%|██████████████▎                                   | 4/14 [05:14<10:16, 61.68s/it]

...

 86%|██████████████████████████████████████████       | 12/14 [10:03<00:52, 26.10s/it]

 93%|█████████████████████████████████████████████▌   | 13/14 [10:25<00:24, 24.86s/it]

100%|█████████████████████████████████████████████████| 14/14 [10:46<00:00, 23.81s/it]

100%|█████████████████████████████████████████████████| 14/14 [10:46<00:00, 46.20s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/T_2003-02.nc


In [18]:
download_MERCATOR(
    Sfiles, "vosaline", starts, ends, x0, x1, y0, y1,outpath+S_out 
)

  0%|                                                          | 0/14 [00:00<?, ?it/s]

  7%|███▌                                              | 1/14 [00:25<05:27, 25.22s/it]

 14%|███████▏                                          | 2/14 [00:46<04:34, 22.86s/it]

 21%|██████████▋                                       | 3/14 [01:07<04:02, 22.03s/it]

 29%|██████████████▎                                   | 4/14 [01:26<03:27, 20.72s/it]

...

 86%|██████████████████████████████████████████       | 12/14 [05:36<00:41, 20.58s/it]

 93%|█████████████████████████████████████████████▌   | 13/14 [05:54<00:19, 19.77s/it]

100%|█████████████████████████████████████████████████| 14/14 [06:11<00:00, 19.05s/it]

100%|█████████████████████████████████████████████████| 14/14 [06:11<00:00, 26.55s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/S_2003-02.nc
